In [ ]:
import urllib.request
import zipfile
import os

zenodo_url = "https://zenodo.org/records/18164998/files/KuaiRec.zip?download=1"
zip_path = "KuaiRec.zip"
extract_folder = "kuairec_data"

print("Dang tai du lieu tu Zenodo...")
urllib.request.urlretrieve(zenodo_url, zip_path)
print("Tai xong. Dang tien hanh giai nen...")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

csv_path = None
for root, dirs, files in os.walk(extract_folder):
    if "small_matrix.csv" in files:
        csv_path = os.path.join(root, "small_matrix.csv").replace('\\', '/')
        break

if csv_path:
    print(f"Da tim thay file tai: {csv_path}")
else:
    print("Khong tim thay file small_matrix.csv")

Dang tai du lieu tu Zenodo...


In [ ]:
con.execute(f'''
    COPY (
        SELECT * FROM read_csv_auto('{csv_path}')
    ) TO 's3://raw/kuairec/small_matrix.parquet' (FORMAT parquet);
''')
print("Da nap KuaiRec vao tang Bronze tren MinIO.")

In [ ]:
profiling_kuairec = con.execute('''
    SELECT 
        COUNT(*) AS tong_tuong_tac,
        SUM(CASE WHEN play_duration < 0 THEN 1 ELSE 0 END) AS loi_play_am,
        SUM(CASE WHEN video_duration <= 0 THEN 1 ELSE 0 END) AS loi_video_hong,
        SUM(CASE WHEN "timestamp" IS NULL THEN 1 ELSE 0 END) AS thieu_timestamp,
        SUM(CASE WHEN watch_ratio > 5.0 THEN 1 ELSE 0 END) AS watch_ratio_bat_thuong
    FROM read_parquet('s3://raw/kuairec/small_matrix.parquet')
''').df()

display(profiling_kuairec)

In [ ]:
con.execute('''
    COPY (
        SELECT 
            user_id,
            video_id,
            play_duration,
            video_duration,
            "time",
            "date",
            "timestamp",
            watch_ratio
        FROM read_parquet('s3://raw/kuairec/small_matrix.parquet')
        WHERE play_duration >= 0 
          AND video_duration > 0
          AND "timestamp" IS NOT NULL
    ) TO 's3://staging/stg_kuairec_interactions.parquet' (FORMAT parquet);
''')
print("Da lam sach va ghi len tang Silver.")

display(con.execute('''
    SELECT user_id, video_id, play_duration, watch_ratio, "timestamp"
    FROM read_parquet('s3://staging/stg_kuairec_interactions.parquet') 
    LIMIT 5
''').df())